Introduction

In this analysis will explore features and find trends in the Goodreads books dataset using mostly SQL queries.
Below I will answer questions, like:
    - what is the most common page count? 
    - Does page count affect the average rating?
    - Which authors have the most books?
I will also try to find hidden gems as well as overhyped books.  

0. Importing libraries and the Goodreads dataset

In [1]:
import pandas as pd
import duckdb

In [ ]:
data = pd.read_csv('books.csv', on_bad_lines='skip')
data = data.copy()
data.columns

Index(['bookID', 'title', 'authors', 'average_rating', 'isbn', 'isbn13',
       'language_code', '  num_pages', 'ratings_count', 'text_reviews_count',
       'publication_date', 'publisher'],
      dtype='object')

In [3]:
duckdb.query("""SELECT COUNT(*) AS number_of_rows FROM data
""").to_df()

,number_of_rows
0,11123


In [4]:
duckdb.query("""
SELECT * 
FROM data
LIMIT 5
""").to_df()

,bookID,title,authors,average_rating,isbn,isbn13,language_code,num_pages,ratings_count,text_reviews_count,publication_date,publisher
0,1,Harry Potter and the Half-Blood Prince (Harry ...,J.K. Rowling/Mary GrandPré,4.57,0439785960,9780439785969,eng,652,2095690,27591,9/16/2006,Scholastic Inc.
1,2,Harry Potter and the Order of the Phoenix (Har...,J.K. Rowling/Mary GrandPré,4.49,0439358078,9780439358071,eng,870,2153167,29221,9/1/2004,Scholastic Inc.
2,4,Harry Potter and the Chamber of Secrets (Harry...,J.K. Rowling,4.42,0439554896,9780439554893,eng,352,6333,244,11/1/2003,Scholastic
3,5,Harry Potter and the Prisoner of Azkaban (Harr...,J.K. Rowling/Mary GrandPré,4.56,043965548X,9780439655484,eng,435,2339585,36325,5/1/2004,Scholastic Inc.
4,8,Harry Potter Boxed Set Books 1-5 (Harry Potte...,J.K. Rowling/Mary GrandPré,4.78,0439682584,9780439682589,eng,2690,41428,164,9/13/2004,Scholastic


1. Data cleaning


In [5]:
data.columns = data.columns.str.strip()

First, let's find null values.

In [6]:
duckdb.query(""" SELECT COUNT(*)          
             FROM data 
             WHERE authors IS NULL 
             OR num_pages IS NULL 
             OR bookID IS NULL 
             OR title IS NULL 
             OR average_rating IS NULL 
             OR isbn IS NULL 
             OR isbn13 IS NULL 
             OR language_code IS NULL 
             OR num_pages IS NULL 
             OR ratings_count IS NULL 
             OR text_reviews_count IS NULL 
             OR publication_date IS NULL 
             OR publisher IS NULL
""").to_df()

,count_star()
0,0


In [7]:
duckdb.query("""
    SELECT COUNT(*)
    FROM data
    WHERE authors = '' 
       OR title = '' 
       OR language_code = ''
       OR publisher = ''
""").to_df()

,count_star()
0,0


In [8]:
duckdb.query("""
    SELECT COUNT(*)
    FROM data
    WHERE bookID = 0
""").to_df()

,count_star()
0,0


Let's also check for books with 0 pages.

In [9]:
duckdb.query("""
    SELECT COUNT(*)
    FROM data
    WHERE num_pages = 0
""").to_df()

,count_star()
0,76


In [10]:
duckdb.query("""SELECT * FROM data
             WHERE num_pages = 0
""").to_df()

,bookID,title,authors,average_rating,isbn,isbn13,language_code,num_pages,ratings_count,text_reviews_count,publication_date,publisher
0,955,The 5 Love Languages / The 5 Love Languages Jo...,Gary Chapman,4.70,0802415318,9780802415318,eng,0,22,4,1/1/2005,Moody Publishers
1,2835,The Tragedy of Pudd'nhead Wilson,Mark Twain/Michael Prichard,3.79,140015068X,9781400150687,eng,0,3,0,1/1/2003,Tantor Media
2,3593,Murder by Moonlight & Other Mysteries (New Adv...,NOT A BOOK,4.00,0743564677,9780743564670,eng,0,7,2,10/3/2006,Simon Schuster Audio
3,3599,The Unfortunate Tobacconist & Other Mysteries ...,NOT A BOOK,3.50,074353395X,9780743533959,eng,0,12,1,10/1/2003,Simon & Schuster Audio
4,4249,The Da Vinci Code (Robert Langdon #2),Dan Brown/Paul Michael,3.84,0739339788,9780739339787,eng,0,91,16,3/28/2006,Random House Audio
...,...,...,...,...,...,...,...,...,...,...,...,...
71,40378,The Chessmen of Mars (Barsoom #5),Edgar Rice Burroughs/John Bolen,3.83,1400130212,9781400130214,eng,0,5147,157,1/1/2005,Tantor Media
72,41273,Fine Lines (One-Eyed Mack #6),Jim Lehrer,3.23,0517164353,9780517164358,eng,0,17,4,11/19/1995,Random House Value Publishing
73,43343,Stowaway and Milk Run: Two Unabridged Stories ...,Mary Higgins Clark/Jan Maxwell,3.49,0671046241,9780671046248,eng,0,64,2,12/1/1999,Simon & Schuster Audio
74,44748,The Mask of the Enchantress,Victoria Holt,3.85,0449210847,9780449210840,eng,0,21,1,10/12/1981,Ivy Books


Let's get rid of these 0 page books by creating a new table that has those filtered out.

In [11]:
duckdb.query(""" CREATE OR REPLACE TABLE books AS SELECT * FROM data WHERE num_pages != 0
""")

Before we move on, if we look closer at above table there, in the authors column there are 'NOT A BOOK' entries, which must be a mistake. Let's see how many of them are left after previous filtering out.

In [12]:
duckdb.query("""SELECT * FROM books WHERE authors = 'NOT A BOOK' OR title ='NOT A BOOK'
             """).to_df()

,bookID,title,authors,average_rating,isbn,isbn13,language_code,num_pages,ratings_count,text_reviews_count,publication_date,publisher
0,19786,The Goon Show Volume 4: My Knees Have Fallen ...,NOT A BOOK,5.00,0563388692,9780563388692,eng,2,3,0,4/1/1996,BBC Physical Audio
1,19787,The Goon Show: Moriarty Where Are You?,NOT A BOOK,4.43,0563388544,9780563388548,eng,2,0,0,3/30/2005,BBC Physical Audio
2,19788,The Goon Show Volume 11: He's Fallen in the W...,NOT A BOOK,5.00,0563388323,9780563388326,eng,2,2,0,10/2/1995,BBC Physical Audio


Looks like there are only 3 of these for 'The Goon Show', published by BBC Physical Audio, which was a comedy radio show. Because of the lack of author's name as well as it not being a book (maybe a transcript of the sketches), we should get rid of that as well.

In [13]:
duckdb.query(""" CREATE OR REPLACE TABLE books AS SELECT * FROM data WHERE authors != 'NOT A BOOK'
""")

Now, let's clear the language codes.

In [14]:
duckdb.query("""SELECT COUNT(*), language_code FROM books GROUP BY language_code """).to_df()

,count_star(),language_code
0,3,enm
1,99,ger
2,1,ara
3,8903,eng
4,1408,en-US
5,144,fre
6,218,spa
7,214,en-GB
8,11,grc
9,7,en-CA


Here we have 27 language codes used in a dataset, but 5 of them are versions of code for english: en-US, en-CA, en-GB, eng and enm for Middle English (a form of english spoken until late 15th century). To simplify thing, I will merge them into one.

In [15]:
duckdb.query(""" SELECT * FROM books WHERE language_code = 'enm'""").to_df()

,bookID,title,authors,average_rating,isbn,isbn13,language_code,num_pages,ratings_count,text_reviews_count,publication_date,publisher
0,2701,The Canterbury Tales (original-spelling edition),Geoffrey Chaucer/Jill Mann,3.49,014042234X,9780140422344,enm,1254,792,59,4/7/2005,Penguin Classics
1,2711,The Riverside Chaucer,Geoffrey Chaucer/Larry Dean Benson/F.N. Robinson,4.18,0395290317,9780395290316,enm,1327,7760,152,12/12/1987,Houghton Mifflin
2,32816,The Canterbury Tales: Fifteen Tales and the Ge...,Geoffrey Chaucer/V.A. Kolve/Glending Olson,3.95,0393925870,9780393925876,enm,600,1149,41,8/1/2005,W. W. Norton & Company


In [16]:
duckdb.query(""" 
UPDATE books 
SET language_code = 'eng'
WHERE language_code IN ('en-US', 'en-CA', 'en-GB', 'enm')
""")

In [17]:
duckdb.query("""SELECT COUNT(*), language_code FROM books GROUP BY language_code """).to_df()

,count_star(),language_code
0,19,mul
1,46,jpn
2,3,lat
3,10,por
4,1,srp
5,1,msa
6,1,glg
7,1,wel
8,2,swe
9,1,nor


Now, let's filter out the titles with less than 50 ratings. This will help with clarity of the data in the analysis.

In [18]:
duckdb.query("""CREATE OR REPLACE TABLE books AS SELECT * FROM books WHERE ratings_count > 50
""")

Let's also find some duplicate titles.

In [19]:
duckdb.query(""" SELECT title , COUNT(*) AS duplicates FROM books GROUP BY title HAVING COUNT(*)>1 ORDER BY duplicates DESC LIMIT 10""").to_df()

,title,duplicates
0,Anna Karenina,7
1,The Odyssey,7
2,The Brothers Karamazov,7
3,The Picture of Dorian Gray,7
4,A Midsummer Night's Dream,6
5,The Secret Garden,6
6,Treasure Island,6
7,Robinson Crusoe,6
8,Jane Eyre,6
9,'Salem's Lot,6


In [20]:
duckdb.query("""SELECT * FROM books WHERE title = 'The Iliad' """).to_df()

,bookID,title,authors,average_rating,isbn,isbn13,language_code,num_pages,ratings_count,text_reviews_count,publication_date,publisher
0,1371,The Iliad,Homer/Robert Fagles/Bernard Knox,3.86,0140275363,9780140275360,eng,683,288792,3423,4/29/1999,Penguin Classics
1,1374,The Iliad,Homer/Robert Fitzgerald/Andrew Ford,3.86,0374529051,9780374529055,eng,588,692,81,4/3/2004,Farrar Straus and Giroux
2,1376,The Iliad,Homer/E.V. Rieu/Peter Jones/D.C.H. Rieu,3.86,0140447946,9780140447941,eng,462,1919,118,1/30/2003,Penguin Classics
3,1377,The Iliad,Homer/W.H.D. Rouse,3.86,0451527372,9780451527370,eng,312,158,15,8/1/1999,Signet Classics
4,22221,The Iliad,Homer,3.86,0471377589,9780471377580,eng,150,3834,134,10/28/1999,John Wiley & Sons
5,32780,The Iliad,Homer/Andrew Lang,3.86,1904633382,9781904633389,eng,542,64,8,9/1/2011,Collector's Library


The outcome poses a certain problem. Like in case of The Illiad, the title appears 6 times, but these are different editions published by different companies. This matters, because these are likely also different translations by different people, containing different forewords, therefore the quality might vary significantly. Bad translation might make a bad book out of a good one. Penguin Classics has two editions of the Illiad, but these are from different years, which might mean different translators or revisited and improved versions ect. Because of this it's valid to leave those as separate titles. Having this in mind, in order to find duplicates we will need to look at titles, authors, language code as well as publication date and publisher, to identify the same edition of the book.

In [21]:
duckdb.query(""" SELECT title, authors, publication_date, publisher, language_code, COUNT(*) AS duplicates
             FROM books 
             GROUP BY title , authors, publication_date, publisher, language_code
             HAVING COUNT(*)>1""").to_df()

,title,authors,publication_date,publisher,language_code,duplicates
0,Cien años de soledad,Gabriel García Márquez,2/7/2006,Plaza y Janes,spa,2
1,Swan Song,Robert R. McCammon,6/1/1987,Pocket Books,eng,2
2,'Salem's Lot,Stephen King/Ron McLarty,1/19/2004,Simon & Schuster Audio,eng,2


In [22]:
duckdb.query(""" SELECT * FROM books 
             WHERE title IN ('Swan Song')  """).to_df()

,bookID,title,authors,average_rating,isbn,isbn13,language_code,num_pages,ratings_count,text_reviews_count,publication_date,publisher
0,11541,Swan Song,Robert R. McCammon,4.28,067162413X,9780671624132,eng,956,147,23,6/1/1987,Pocket Books
1,11557,Swan Song,Robert R. McCammon,4.28,0671741039,9780671741037,eng,956,46244,2540,6/1/1987,Pocket Books


Interestingly, those duplicates would not appear if we tried to find them the usual way. If we look at 'Swan Song' by Robert R. McCammon, we can notice that there are two 'Pocket Books' editions published the same date 6/1/1987 with the same score 4.28, but different text reviews count, isbn code and ratings count, which escapes the usual duplicate row check.

In [23]:
duckdb.query("""
    CREATE OR REPLACE TABLE books AS
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (
                   PARTITION BY title, authors, language_code, publication_date, publisher
                   ORDER BY bookID
               ) AS row_num
        FROM books
    )
    WHERE row_num = 1
""")

In [24]:
duckdb.query("""
    CREATE OR REPLACE TABLE books AS
    SELECT * EXCLUDE (row_num) FROM books
""")

In [25]:
duckdb.query(""" SELECT title , authors, publication_date, publisher, language_code, COUNT(*) AS duplicates
             FROM books 
             GROUP BY title , authors, publication_date, publisher, language_code
             HAVING COUNT(*)>1""").to_df()

,title,authors,publication_date,publisher,language_code,duplicates


Let's also see if there are any ratings that are not in a default 'out of 5' mode.

In [26]:
duckdb.query(""" SELECT * FROM books WHERE average_rating > 5""").to_df()

,bookID,title,authors,average_rating,isbn,isbn13,language_code,num_pages,ratings_count,text_reviews_count,publication_date,publisher


2. Dataset Exploration

2.1 How many books are there in each language and what are their ratings?

In [27]:
duckdb.query(""" SELECT language_code, COUNT(*)book_count, MIN(average_rating), MAX(average_rating), AVG(average_rating), MEDIAN(average_rating) 
             FROM books 
             GROUP BY language_code 
             ORDER BY  COUNT(*) DESC""").to_df()

,language_code,book_count,min(average_rating),max(average_rating),avg(average_rating),median(average_rating)
0,eng,8958,2.40,4.82,3.950577,3.960
1,spa,116,3.28,4.57,3.954397,3.955
2,fre,65,3.46,4.44,3.965692,3.970
3,ger,23,3.54,4.56,3.991304,3.990
4,mul,10,3.85,4.40,4.163000,4.225
5,jpn,10,4.11,4.50,4.292000,4.240
6,por,8,3.89,4.09,3.986250,3.980
7,grc,4,3.70,4.21,3.962500,3.970
8,ita,3,3.89,4.02,3.963333,3.980
9,lat,3,4.17,4.47,4.353333,4.420


2.2 Average rating distribution across the whole dataset

In [28]:
duckdb.query(""" SELECT  MIN(average_rating) AS lowest_rating,
              MAX(average_rating) AS highest_rating, 
             AVG(average_rating) AS average_rating, 
             MEDIAN(average_rating) AS median_of_rating
             FROM books
             """).to_df()

,lowest_rating,highest_rating,average_rating,median_of_rating
0,2.4,4.82,3.951803,3.96


Average rating distribution visualization in Tableau: https://public.tableau.com/app/profile/bartek.owsiany/viz/GoodreadsBooks-AverageRatingDistribution/Sheet1 


2.3 What's the most common page count range and what are average ratings in those ranges?

I will create 5 ranges for page counts. It is difficult to establish the ranges of those page counts, since something like 'medium length book' is not a clear cut category. Nevertheless, this can still reveal some insights.

In [29]:
duckdb.query("""
    SELECT 
        CASE 
            WHEN num_pages BETWEEN 1 AND 100 THEN 'articles/short_story (1-100)'
            WHEN num_pages BETWEEN 101 AND 200 THEN 'short_books (101-200)'
            WHEN num_pages BETWEEN 201 AND 400 THEN 'medium_length_books (201-400)'
            WHEN num_pages BETWEEN 401 AND 700 THEN 'long_books (401-700)'
            ELSE 'very_long_books (above 700)'
        END AS book_length,
        COUNT(*) AS book_count,
        ROUND(AVG(average_rating), 2) AS avg_rating
    FROM books
    GROUP BY book_length
    ORDER BY book_count DESC
""").to_df()

,book_length,book_count,avg_rating
0,medium_length_books (201-400),4457,3.91
1,long_books (401-700),1994,3.99
2,short_books (101-200),1502,3.93
3,articles/short_story (1-100),633,3.99
4,very_long_books (above 700),623,4.14


Page count ranges visualization in Tableau:
https://public.tableau.com/app/profile/bartek.owsiany/viz/GoodreadsBooks-Pageranges/Dashboard1

As we can see, the medium length book are by far the most common, but not the most popular. Surprisingly, the most popular books are those counting more than 700 pages (very long books) - although, unexpected, it's not completely shocking, since the average ratings do not vary by much. This result might suggest that people like to commit to the long novels, but overall the page count does not have big effect on rating.

3. Analysis

3.1 Authors with the most books

In [30]:
duckdb.query(""" SELECT authors, COUNT(*) AS number_of_books
            FROM books
            GROUP BY authors
            ORDER BY number_of_books DESC
            LIMIT 10
            """).to_df()

,authors,number_of_books
0,Stephen King,36
1,P.G. Wodehouse,35
2,Orson Scott Card,32
3,Agatha Christie,31
4,Mercedes Lackey,29
5,Piers Anthony,28
6,Dick Francis,27
7,Sandra Brown,26
8,Terry Pratchett,23
9,Rumiko Takahashi,22


3.2 Authors with the highest average rating and minimum 3 books

In [31]:
duckdb.query(""" SELECT authors, ROUND(AVG(average_rating) , 2) AS avg_rating, COUNT(*) AS book_count
             FROM books
             GROUP BY authors
             HAVING COUNT(*) >= 3
             ORDER BY avg_rating DESC
             LIMIT 10
             """).to_df()

,authors,avg_rating,book_count
0,Bill Watterson,4.71,7
1,Hayao Miyazaki/Matt Thorn/Kaori Inoue/Joe Yama...,4.61,3
2,Hiromu Arakawa/Akira Watanabe,4.57,12
3,J.K. Rowling/Mary GrandPré,4.55,6
4,J.K. Rowling,4.53,8
5,Joseph Frank,4.50,3
6,Kyoko Hikawa/Yuko Sawada/Freeman Wong,4.47,3
7,Warren Ellis/Darick Robertson/Rodney Ramos,4.46,4
8,Chie Shinohara,4.45,4
9,Karen Kingsbury/Gary Smalley,4.44,5


3.3 Hidden gems analysis

The goal of hidden gems search is to find titles that have low amount of ratings, but high average rating score. The problem here is that we can't just take a lowest rating count with the highest rating score, because having a book with one five star review does not constitute a hidden gem. It just might be so that it appeals to this one person, but it wouldn't necessarily be appreciated by anyone else, or the reviewer might have happened to be a friend of an author and so on. This is simply not reliable data. Therefore we have to establish a filter for minimal amount of reviews and then find the highest rated titles. Previously we filtered out all the book with 50 or less ratings. Let's now see the range we are working with.

In [32]:
duckdb.query(""" (SELECT title, ratings_count 
             FROM books 
             ORDER BY ratings_count LIMIT 1) 
             union all
             (SELECT title, ratings_count FROM books ORDER BY ratings_count DESC LIMIT 1)""").to_df()

,title,ratings_count
0,A Chosen Few: The Resurrection of European Jew...,51
1,Twilight (Twilight #1),4597666


In that range, 51 still seems low and slightly arbitrary. Instead of that let's just use quantiles.

In [33]:
duckdb.query(""" SELECT * FROM books 
             WHERE ratings_count BETWEEN (SELECT QUANTILE_CONT(ratings_count, 0.25) FROM books)
             AND (SELECT QUANTILE_CONT(ratings_count, 0.45) FROM books)
             ORDER BY average_rating DESC
             LIMIT 20""").to_df()

,bookID,title,authors,average_rating,isbn,isbn13,language_code,num_pages,ratings_count,text_reviews_count,publication_date,publisher
0,44826,The Price of the Ticket: Collected Nonfiction ...,James Baldwin,4.70,0312643063,9780312643065,eng,712,404,30,9/15/1985,St. Martin's Press
1,26805,The Sibley Field Guide to Birds of Western Nor...,David Allen Sibley,4.69,0679451218,9780679451211,eng,473,730,36,4/29/2003,Alfred A. Knopf
2,17280,The Feynman Lectures on Physics Vol 3,Richard P. Feynman/Robert B. Leighton/Matthew ...,4.63,0805390499,9780805390490,eng,384,716,15,9/1/2005,Addison Wesley Publishing Company
3,13206,The Collected Autobiographies of Maya Angelou,Maya Angelou,4.63,0679643257,9780679643258,eng,1184,991,55,9/21/2004,Modern Library
4,17334,The Complete C.S. Lewis Signature Classics,C.S. Lewis,4.61,0061208493,9780061208492,eng,746,925,40,2/6/2007,HarperOne
5,3529,The World's First Love: Mary Mother of God,Fulton J. Sheen,4.59,0898705975,8987059752,eng,276,641,63,9/1/1996,Ignatius Press
6,33342,The More Than Complete Hitchhiker's Guide (Hit...,Douglas Adams,4.58,0681403225,9780681403222,eng,624,433,34,11/1/1989,Longmeadow Press
7,16602,Coffin: The Art of Vampire Hunter D,Yoshitaka Amano,4.58,1595820612,9781595820617,eng,199,596,16,11/1/2006,Dark Horse Books
8,41908,Harry Potter und der Gefangene von Askaban (Ha...,J.K. Rowling/Rufus Beck,4.56,3895849618,9783895849619,ger,13,313,8,8/31/2002,Dhv der Hörverlag
9,31692,The Complete Novels of Jane Austen,Jane Austen,4.55,1840220554,9781840220551,eng,1431,924,28,1/5/2005,Wordsworth Editions


There is something alarming about those outcomes. It is difficult to call anything written by J.R.R. Tolkien, J.K. Rowling, William Shakespeare or C.S. Lewis as a hidden gem. These results indicate a deeper problem with the data, e.g. different editions of the same title or the same title but in a different language (as in the case of "Harry Potter und der Gefangene von Askaban"). The problem complicates when we recall previous conclusion about the different translations and editions not being equal in quality, therefore treating them as separate book! The way we can attempt to solve this difficult problem is to perhaps find a list of most popular authors and filter it out from the above table. In order to do that, we will have to do a kind of partial inversion of the previous query, where we looked for lower than 50th quantile of the ratings_count to find those less known authors and titles. Now we have to look at higher numbers in ratings_count as it should account for the popularity (or at least familiarity) of the author.

In [34]:
top_authors = duckdb.query(""" SELECT authors, SUM(ratings_count) AS number_of_ratings 
                            FROM books
                            GROUP BY authors
                            ORDER BY number_of_ratings DESC
                            LIMIT 20 """).to_df()

In [35]:
top_authors

,authors,number_of_ratings
0,J.K. Rowling/Mary GrandPré,8923980.0
1,J.R.R. Tolkien,4776638.0
2,Stephenie Meyer,4597666.0
3,Dan Brown,4135380.0
4,Nicholas Sparks,3048149.0
5,Stephen King,2985555.0
6,J.D. Salinger,2777908.0
7,Rick Riordan,2413447.0
8,George Orwell/Boris Grabnar/Peter Škerl,2111750.0
9,John Steinbeck,2091507.0


Above we have a list of 20 most rated authors - which should correspond with their popularity or familiarity. The problem that emerges here is that we cannot just filter out J.K. Rowling because, as we can see in the table, her name appears in form of 'J.K. Rowling/Mary GrandPré' and in the previous outcome we had 'J.K. Rowling/Rufus Beck' for the german edition of Harry Potter. So in that case referring or filtering out 'J.K. Rowling' is not the same as filtering out 'J.K. Rowling/Mary GrandPré'. Same for the other authors like 'George Orwell/Boris Grabnar/Peter Škerl' or 'William Shakespeare/Paul Werstine/Barbara A. M' as well as many other. This complicates the issue, since the same author can be referenced in many different ways, and we can't simply look through 11123 rows to identify all the combinations of names with their referents. This brings me to employment of an admittedly not very elegant solution. 

Below I used a python nested loop, which extracts names from the previous table and splits those that appear in one cell with '/' sign, so instead of 'J.K. Rowling/Mary GrandPré' we get 'J.K. Rowling' and 'Mary GrandPré' as separate names. After we have our names extracted, each of them is put in quotations and surrounded by '%' signs, followed by "AND authors NOT LIKE". This format allows SQL to treat the names as wildcards, meaning that every row containing letter combination of 'J.K. Rowling' in authors column will be filtered out regardless of whether there are any other names in that cell.

This generates somewhat long and repetitive query that can be copy and paste below in order to filter out unwanted names. The python loop just does the tedious and repetitive work of extracting and formatting the names into SQL query for us.

In [36]:
author_list = []
for x in top_authors['authors']:
    x = x.split('/')
    for name in x:
        name = "'" + "%" + name + "%" + "'" + "AND authors NOT LIKE "
        print(name)

'%J.K. Rowling%'AND authors NOT LIKE 
'%Mary GrandPré%'AND authors NOT LIKE 
'%J.R.R. Tolkien%'AND authors NOT LIKE 
'%Stephenie Meyer%'AND authors NOT LIKE 
'%Dan Brown%'AND authors NOT LIKE 
'%Nicholas Sparks%'AND authors NOT LIKE 
'%Stephen King%'AND authors NOT LIKE 
'%J.D. Salinger%'AND authors NOT LIKE 
'%Rick Riordan%'AND authors NOT LIKE 
'%George Orwell%'AND authors NOT LIKE 
'%Boris Grabnar%'AND authors NOT LIKE 
'%Peter Škerl%'AND authors NOT LIKE 
'%John Steinbeck%'AND authors NOT LIKE 
'%Jodi Picoult%'AND authors NOT LIKE 
'%William Golding%'AND authors NOT LIKE 
'%William Shakespeare%'AND authors NOT LIKE 
'%Paul Werstine%'AND authors NOT LIKE 
'%Barbara A. Mowat%'AND authors NOT LIKE 
'%Lois Lowry%'AND authors NOT LIKE 
'%John Grisham%'AND authors NOT LIKE 
'%Roald Dahl%'AND authors NOT LIKE 
'%Quentin Blake%'AND authors NOT LIKE 
'%James Patterson%'AND authors NOT LIKE 
'%Paulo Coelho%'AND authors NOT LIKE 
'%Alan R. Clarke%'AND authors NOT LIKE 
'%Özdemir İnce%'AND aut

In [37]:
hidden_gems = duckdb.query(""" SELECT * FROM books 
WHERE ratings_count BETWEEN (SELECT QUANTILE_CONT(ratings_count, 0.25) FROM books)
AND (SELECT QUANTILE_CONT(ratings_count, 0.45) FROM books)
AND authors NOT LIKE '%Arthur Golden%'AND authors
              NOT LIKE '%J.K. Rowling%'AND authors NOT LIKE '%Mary GrandPré%'
             AND authors NOT LIKE '%J.R.R. Tolkien%'AND authors NOT LIKE '%Stephenie Meyer%'
             AND authors NOT LIKE '%Dan Brown%'AND authors NOT LIKE '%Nicholas Sparks%'
             AND authors NOT LIKE '%Stephen King%'AND authors NOT LIKE '%J.D. Salinger%'
             AND authors NOT LIKE '%Rick Riordan%'AND authors NOT LIKE '%George Orwell%'
             AND authors NOT LIKE '%Boris Grabnar%'AND authors NOT LIKE '%Peter Škerl%'
             AND authors NOT LIKE '%John Steinbeck%'AND authors NOT LIKE '%Jodi Picoult%'
             AND authors NOT LIKE '%William Golding%'AND authors NOT LIKE '%William Shakespeare%'
             AND authors NOT LIKE '%Paul Werstine%'AND authors NOT LIKE '%Barbara A. Mowat%'
             AND authors NOT LIKE '%Lois Lowry%'AND authors NOT LIKE '%John Grisham%'
             AND authors NOT LIKE '%Roald Dahl%'AND authors NOT LIKE '%Quentin Blake%'
             AND authors NOT LIKE '%James Patterson%'AND authors NOT LIKE '%Paulo Coelho%'
             AND authors NOT LIKE '%Alan R. Clarke%'AND authors NOT LIKE '%Özdemir İnce%'
             AND authors NOT LIKE '%Agatha Christie%'AND authors NOT LIKE '%Arthur Golden%'
ORDER BY average_rating DESC
LIMIT 20""").to_df()

In [38]:
hidden_gems

,bookID,title,authors,average_rating,isbn,isbn13,language_code,num_pages,ratings_count,text_reviews_count,publication_date,publisher
0,44826,The Price of the Ticket: Collected Nonfiction ...,James Baldwin,4.70,0312643063,9780312643065,eng,712,404,30,9/15/1985,St. Martin's Press
1,26805,The Sibley Field Guide to Birds of Western Nor...,David Allen Sibley,4.69,0679451218,9780679451211,eng,473,730,36,4/29/2003,Alfred A. Knopf
2,13206,The Collected Autobiographies of Maya Angelou,Maya Angelou,4.63,0679643257,9780679643258,eng,1184,991,55,9/21/2004,Modern Library
3,17280,The Feynman Lectures on Physics Vol 3,Richard P. Feynman/Robert B. Leighton/Matthew ...,4.63,0805390499,9780805390490,eng,384,716,15,9/1/2005,Addison Wesley Publishing Company
4,17334,The Complete C.S. Lewis Signature Classics,C.S. Lewis,4.61,0061208493,9780061208492,eng,746,925,40,2/6/2007,HarperOne
5,3529,The World's First Love: Mary Mother of God,Fulton J. Sheen,4.59,0898705975,8987059752,eng,276,641,63,9/1/1996,Ignatius Press
6,33342,The More Than Complete Hitchhiker's Guide (Hit...,Douglas Adams,4.58,0681403225,9780681403222,eng,624,433,34,11/1/1989,Longmeadow Press
7,16602,Coffin: The Art of Vampire Hunter D,Yoshitaka Amano,4.58,1595820612,9781595820617,eng,199,596,16,11/1/2006,Dark Horse Books
8,31692,The Complete Novels of Jane Austen,Jane Austen,4.55,1840220554,9781840220551,eng,1431,924,28,1/5/2005,Wordsworth Editions
9,31672,The Complete Novels,Jane Austen/Karen Joy Fowler,4.55,0143039504,9780143039501,eng,1278,425,39,3/28/2006,Penguin Classics


Although the solution is a bit clunky, it yielded results. In top 20 we don't see J.K. Rowling, George Orwell or William Shakespeare, instead we have books with hundreds of reviews and well above the average rating score, which very well might constitute a hidden gem list. 

Surprisingly, there are still some very famous names in the table like C.S. Lewis, Marcel Proust or Jane Austen. By closer inspection, we can notice that many of those spaces are occupied by compiled works. It is up to debate whether we should treat those as a single book or not. Perhaps the way to treat it should be determined by our goals in mind. If we were interested in particular titles rather than compilations we could simply filter out the titles containing key words like 'Complete' and 'Collected'. This solution is a double edged sword, since it might filter out some authors worth attention as well as books containing those key words that happened not to be a compilations. On the other hand, some of those compilations might contain some classic novels that are too popular to be on that list and that compilation would not appear in the ranking otherwise.

In [39]:
duckdb.query("""SELECT * FROM books 
WHERE ratings_count BETWEEN (SELECT QUANTILE_CONT(ratings_count, 0.25) FROM books)
AND (SELECT QUANTILE_CONT(ratings_count, 0.45) FROM books)
AND authors NOT LIKE '%Arthur Golden%'AND authors
              NOT LIKE '%J.K. Rowling%'AND authors NOT LIKE '%Mary GrandPré%'
             AND authors NOT LIKE '%J.R.R. Tolkien%'AND authors NOT LIKE '%Stephenie Meyer%'
             AND authors NOT LIKE '%Dan Brown%'AND authors NOT LIKE '%Nicholas Sparks%'
             AND authors NOT LIKE '%Stephen King%'AND authors NOT LIKE '%J.D. Salinger%'
             AND authors NOT LIKE '%Rick Riordan%'AND authors NOT LIKE '%George Orwell%'
             AND authors NOT LIKE '%Boris Grabnar%'AND authors NOT LIKE '%Peter Škerl%'
             AND authors NOT LIKE '%John Steinbeck%'AND authors NOT LIKE '%Jodi Picoult%'
             AND authors NOT LIKE '%William Golding%'AND authors NOT LIKE '%William Shakespeare%'
             AND authors NOT LIKE '%Paul Werstine%'AND authors NOT LIKE '%Barbara A. Mowat%'
             AND authors NOT LIKE '%Lois Lowry%'AND authors NOT LIKE '%John Grisham%'
             AND authors NOT LIKE '%Roald Dahl%'AND authors NOT LIKE '%Quentin Blake%'
             AND authors NOT LIKE '%James Patterson%'AND authors NOT LIKE '%Paulo Coelho%'
             AND authors NOT LIKE '%Alan R. Clarke%'AND authors NOT LIKE '%Özdemir İnce%'
             AND authors NOT LIKE '%Agatha Christie%'AND authors NOT LIKE '%Arthur Golden%'
             AND title NOT LIKE '%Collected%'
             AND title NOT LIKE '%Complete%' 
             ORDER BY average_rating DESC
             LIMIT 20
             """).to_df()

,bookID,title,authors,average_rating,isbn,isbn13,language_code,num_pages,ratings_count,text_reviews_count,publication_date,publisher
0,26805,The Sibley Field Guide to Birds of Western Nor...,David Allen Sibley,4.69,0679451218,9780679451211,eng,473,730,36,4/29/2003,Alfred A. Knopf
1,17280,The Feynman Lectures on Physics Vol 3,Richard P. Feynman/Robert B. Leighton/Matthew ...,4.63,0805390499,9780805390490,eng,384,716,15,9/1/2005,Addison Wesley Publishing Company
2,3529,The World's First Love: Mary Mother of God,Fulton J. Sheen,4.59,0898705975,8987059752,eng,276,641,63,9/1/1996,Ignatius Press
3,16602,Coffin: The Art of Vampire Hunter D,Yoshitaka Amano,4.58,1595820612,9781595820617,eng,199,596,16,11/1/2006,Dark Horse Books
4,28395,Remembrance of Things Past: Volume II - The Gu...,Marcel Proust/C.K. Scott Moncrieff/Terence Kil...,4.53,0394711831,9780394711836,eng,1216,866,42,8/27/1982,Vintage
5,13674,The Life and Games of Mikhail Tal,Mikhail Tal/Iakov Damsky/Kenneth P. Neat,4.52,1857442024,9781857442021,eng,496,442,24,7/1/1997,Everyman Chess
6,23319,Transcending the Levels of Consciousness: The ...,David R. Hawkins,4.51,0971500746,9780971500747,eng,407,461,29,6/28/2006,Veritas Publishing
7,8362,Animal: The Definitive Visual Guide to the Wor...,David Burnie/Don E. Wilson,4.51,0756616344,9780756616342,eng,624,570,17,9/19/2005,DK
8,13551,From Far Away Vol. 14,Kyoko Hikawa/Yuko Sawada/Freeman Wong,4.50,142150541X,9781421505411,eng,208,986,37,1/9/2007,VIZ Media
9,11431,The Red Gloves Collection (Red Gloves #1-4),Karen Kingsbury,4.49,0446579629,9780446579629,eng,640,718,36,10/10/2006,FaithWords


3.4 Overhyped books analysis

The goal of finding the overhyped books is to find titles with very high ratings count but below average rating score. So let's first remind what is the  average rating score.

In [40]:
duckdb.query(""" SELECT ROUND(AVG(average_rating), 2) FROM books """).to_df()

,"round(avg(average_rating), 2)"
0,3.95


Now we can search for the books in higher percentile of ratings count and lower than 3.95 average rating score. 

In [41]:
lowest_rated100 = duckdb.query(""" SELECT title, authors, average_rating, ratings_count FROM books 
             WHERE ratings_count BETWEEN (SELECT QUANTILE_CONT(ratings_count, 0.75) FROM books)
             AND (SELECT QUANTILE_CONT(ratings_count, 1.0) FROM books)
             AND average_rating < 3.95
             ORDER BY average_rating
             LIMIT 100""").to_df()

In [42]:
lowest_rated100.head(10)

,title,authors,average_rating,ratings_count
0,Four Blondes,Candace Bushnell,2.82,23409
1,Lost,Gregory Maguire/Douglas Smith,2.82,13152
2,The Jane Austen Book Club,Karen Joy Fowler,3.08,57720
3,Billy Budd Sailor,Herman Melville,3.12,12599
4,The Mermaid Chair,Sue Monk Kidd,3.13,68363
5,The Autograph Man,Zadie Smith,3.16,9541
6,The Castle of Otranto,Horace Walpole,3.18,16370
7,Songs of the Humpback Whale,Jodi Picoult,3.20,27590
8,Falling Man,Don DeLillo,3.21,10304
9,The Rule of Four,Ian Caldwell/Dustin Thomason,3.23,25111


These titles have the lowest ratings, but not necessarily highest rating count in the range. Let's also see the titles with the most ratings count while having below the average rating score.

In [43]:
most_ratings100 = duckdb.query(""" SELECT title, authors, average_rating, ratings_count FROM books 
             WHERE ratings_count BETWEEN (SELECT QUANTILE_CONT(ratings_count, 0.75) FROM books)
             AND (SELECT QUANTILE_CONT(ratings_count, 1.0) FROM books)
             AND average_rating < 3.95
             ORDER BY ratings_count DESC
             LIMIT 100""").to_df()

In [44]:
most_ratings100.head(10)

,title,authors,average_rating,ratings_count
0,Twilight (Twilight #1),Stephenie Meyer,3.59,4597666
1,The Catcher in the Rye,J.D. Salinger,3.80,2457092
2,Angels & Demons (Robert Langdon #1),Dan Brown,3.89,2418736
3,Animal Farm,George Orwell/Boris Grabnar/Peter Škerl,3.93,2111750
4,Lord of the Flies,William Golding,3.68,2036679
5,Romeo and Juliet,William Shakespeare/Paul Werstine/Barbara A. M...,3.74,1893917
6,Of Mice and Men,John Steinbeck,3.87,1755253
7,The Da Vinci Code (Robert Langdon #2),Dan Brown,3.84,1679706
8,The Alchemist,Paulo Coelho/Alan R. Clarke/Özdemir İnce,3.86,1631221
9,Eat Pray Love,Elizabeth Gilbert,3.55,1362264


Now having both of tables we can see the overlap.

In [45]:
duckdb.query(""" SELECT lowest_rated100.*
             FROM lowest_rated100
             INNER JOIN most_ratings100 
             ON lowest_rated100.title = most_ratings100.title""").to_df()

,title,authors,average_rating,ratings_count
0,The Scarlet Letter,Nathaniel Hawthorne/Thomas E. Connolly/Nina Baym,3.40,609586
1,Heart of Darkness,Joseph Conrad,3.42,321078
2,The Pearl,John Steinbeck,3.46,160706
3,The Canterbury Tales,Geoffrey Chaucer/Nevill Coghill,3.49,168559
4,Gerald's Game,Stephen King,3.51,117178
5,Men Are from Mars Women Are from Venus,John Gray,3.55,132062
6,Eat Pray Love,Elizabeth Gilbert,3.55,1362264


Although quite controversial, above we have a top 9 most overrated books. It almost feels wrong to call some of these titles overrated. It might be because the book is not unequivocally bad, but rather divisive, creating radically opposite reactions - this does not necessarily mean 'overhyped'. In order to truly answer this question, we would need a specific data about ratings of each book, as well as perhaps a deep dive into a written reviews and this task goes beyond this analysis and the content of the dataset.

4. Concluding remarks

The goal of the following analysis was to explore different aspects and discover trends in the Goodreads dataset.

1. Data quality and its challenges

Finding null values proved to be slightly more challenging than just using ‘IS NULL’ in a query, since this didn’t yield any results. It was necessary to check for zero values as well (like 0 page books ect.), which turned out to be the right choice, exposing at the same time the ‘NOT A BOOK’ entries, which also had to be deleted. 

2. The edition problem

Finding duplicate titles also turned out to be more complicated than expected, exposing the problem with different editions. Different edition of the same title might have different translations, preface ect., which has a significant influence over the quality of the book. Although there were duplicates, they couldn’t be discovered the usual way, since they differed by text reviews count, isbn code and ratings count but not everything else which made them the same edition of the same title. So what had to be established as constituting a duplicate was having the same title, author, language code, publisher and publication date.

3. Page count doesn't dramatically affect ratings

We found that books over 700 pages in length received the highest average rating, but the overall scores did not differ significantly across page ranges. This would suggest that length of the book does not strongly affect the average score. 

4. Hidden gems are hidden

Hidden gems section turned out to be the most challenging with authors names appearing in combinations with names of editors or translators ect., making it troublesome to create a query filtering out the popular authors from the hidden gems table. Different editions of famous books were finding their ways to the list that supposed to contain unknown titles, even under disguise of collected works.

5. Complicated nature of ‘overhyped’

Finding overhyped book was not as difficult as some other tasks, but some reflection might undermine this result. In order to truly find the overhyped titles, we would have to look into ratings of each book to see if it is indeed overhyped or is it just very divisive. These two seem to be very different, as first would be mostly negative or neutral reviews, not living up to the publicity - that would correspond for example with the book being boring. The other would mean that reactions are both very positive as well as very negative - that might mean something completely opposite, like depending on your taste you will either love it or hate it. In the second case the publicity stems from the extreme reactions that book elicits, and not, for example successful marketing campaign.